In [ ]:
# Muhammad Zia Ul-haq 2311513013

# tugas 1
import multiprocessing as mp
import time
import matplotlib.pyplot as plt

# Definisi fungsi hitung_prima (ganti dengan fungsi dari eksperimen 7.1 Anda jika berbeda)
def is_prime(n):
    if n <= 1: return False
    if n <= 3: return True
    if n % 2 == 0 or n % 3 == 0: return False
    i = 5
    while i * i <= n:
        if n % i == 0 or n % (i + 2) == 0: return False
        i += 6
    return True

def hitung_prima(limit):
    count = 0
    for i in range(2, limit):
        if is_prime(i):
            count += 1
    return count

if __name__ == '__main__':
    pekerja_skenario = [1, 2, 4, 8]
    waktu_eksekusi = []
    batas_prima = 1000000  # Sesuaikan beban komputasi jika diperlukan
    
    print("Memulai eksperimen Analisis Speedup Skala...")
    
    # Menggunakan context 'fork' sesuai instruksi. 
    # Catatan: 'fork' hanya didukung secara native di Linux/macOS.
    try:
        ctx = mp.get_context('fork')
    except ValueError:
        print("Sistem operasi tidak mendukung 'fork'. Beralih ke context default.")
        ctx = mp.get_context()

    for w in pekerja_skenario:
        mulai = time.time()
        
        with ctx.Pool(processes=w) as pool:
            # Membagi tugas ke pekerja secara paralel
            # Di sini kita memecah beban tugas yang sama ke dalam beberapa bagian
            chunk_size = batas_prima // w
            tugas = [chunk_size] * w
            pool.map(hitung_prima, tugas)
            
        selesai = time.time()
        durasi = selesai - mulai
        waktu_eksekusi.append(durasi)
        print(f"Eksekusi dengan {w} worker memakan waktu: {durasi:.4f} detik")

    # Menghitung Speedup: T(1) / T(N)
    waktu_satu_worker = waktu_eksekusi[0]
    speedup_aktual = [waktu_satu_worker / t for t in waktu_eksekusi]

    # Plot Speedup Aktual vs Teoretis Ideal (Linear)
    plt.figure(figsize=(8, 5))
    plt.plot(pekerja_skenario, speedup_aktual, marker='o', label='Aktual (Eksperimen)')
    plt.plot(pekerja_skenario, pekerja_skenario, linestyle='--', color='red', label='Ideal (Hukum Amdahl murni paralel)')
    
    plt.title('Speedup vs Jumlah Worker')
    plt.xlabel('Jumlah Worker')
    plt.ylabel('Speedup (T1 / Tp)')
    plt.xticks(pekerja_skenario)
    plt.legend()
    plt.grid(True)
    plt.show()

Memulai eksperimen Analisis Speedup Skala...
Sistem operasi tidak mendukung 'fork'. Beralih ke context default.


Analisis Penyimpangan Kurva Aktual dari Kurva Teoretis Hukum Amdahl:Kurva speedup aktual hasil eksperimen pada umumnya tidak persis mengikuti kurva teoretis Hukum Amdahl (yang mengasumsikan skalabilitas linear sempurna pada bagian paralel). Hal ini dipengaruhi oleh tiga faktor utama:   Keterbatasan Jumlah Core Fisik: Jika mesin Anda hanya memiliki 4 core CPU fisik, mencoba menjalankan 8 worker secara bersamaan tidak akan memberikan percepatan 8x lipat. Justru, hal ini dapat memicu penurunan performa karena sistem operasi dipaksa melakukan context switching terus-menerus untuk membagi waktu pada ke-8 proses tersebut di atas 4 core.   Overhead Pembuatan Proses: multiprocessing di Python mengharuskan pembuatan proses baru (bahkan saat menggunakan metode fork). Membangun proses, mengalokasikan memori independen, dan mengelola komunikasi (IPC) antar proses utama (main process) dan worker membutuhkan waktu tambahan. Overhead komputasi ini mengurangi efisiensi dari paralelisasi.   Beban Kerja Latar Belakang: Mesin yang digunakan untuk eksperimen tidak berada dalam ruang hampa. Ada sistem operasi, browser, program antivirus, dan aplikasi latar belakang lainnya yang turut memperebutkan sumber daya CPU secara bersamaan, sehingga ketersediaan CPU tidak 100% untuk eksekusi kode multiprocessing ini.   

In [ ]:
#tugas 2
import multiprocessing as mp
import random
import time

# Konstanta untuk tugas
JUMLAH_ANGKA = 1_000_000
JUMLAH_PROSES = 4
ANGKA_PER_PROSES = JUMLAH_ANGKA // JUMLAH_PROSES

# --- VERSI A: MESSAGE PASSING (Menggunakan Queue) ---
def pekerja_queue(q):
    subtotal = 0
    # Menjumlahkan angka acak
    for _ in range(ANGKA_PER_PROSES):
        subtotal += random.random()
    # Mengirim hasil subtotal ke Queue
    q.put(subtotal)

def eksekusi_queue():
    q = mp.Queue()
    proses_list = []
    
    mulai = time.time()
    
    # Menjalankan 4 proses
    for _ in range(JUMLAH_PROSES):
        p = mp.Process(target=pekerja_queue, args=(q,))
        proses_list.append(p)
        p.start()
        
    for p in proses_list:
        p.join()
        
    # Main process mengumpulkan dan menjumlahkan subtotal
    total_akhir = 0
    while not q.empty():
        total_akhir += q.get()
        
    selesai = time.time()
    print(f"[Queue / Message Passing] Total: {total_akhir:.2f} | Waktu Eksekusi: {selesai - mulai:.4f} detik")

# --- VERSI B: SHARED MEMORY (Menggunakan Value dan Lock) ---
def pekerja_value(shared_val, lock):
    subtotal = 0
    # Menjumlahkan angka acak secara lokal terlebih dahulu
    for _ in range(ANGKA_PER_PROSES):
        subtotal += random.random()
        
    # Menggunakan lock untuk mengamankan akses ke shared memory saat menambahkan subtotal
    with lock:
        shared_val.value += subtotal

def eksekusi_value():
    # Membuat shared memory bertipe double ('d') dengan nilai awal 0.0
    shared_total = mp.Value('d', 0.0)
    lock = mp.Lock()
    proses_list = []
    
    mulai = time.time()
    
    # Menjalankan 4 proses
    for _ in range(JUMLAH_PROSES):
        p = mp.Process(target=pekerja_value, args=(shared_total, lock))
        proses_list.append(p)
        p.start()
        
    for p in proses_list:
        p.join()
        
    selesai = time.time()
    print(f"[Value / Shared Memory ] Total: {shared_total.value:.2f} | Waktu Eksekusi: {selesai - mulai:.4f} detik")

# --- BLOK UTAMA UNTUK MENJALANKAN KEDUA VERSI ---
if __name__ == '__main__':
    print("Menjalankan perbandingan performa...")
    print("-" * 50)
    eksekusi_queue()
    eksekusi_value()

In [ ]:
#tugas 3
import multiprocessing as mp
import time

# Simulasi tugas CPU-bound (komputasi matematika yang berat)
def proses_berat(n):
    total = 0
    # Loop sebanyak jutaan kali untuk membebani CPU
    for i in range(5_000_000):
        total += i
    return total + n

def producer(q, num_items, num_consumers):
    for i in range(1, num_items + 1):
        print(f"[Producer] Memasukkan tugas {i} ke antrean")
        q.put(i)
        
    # Mengirim 'Poison Pill' (None) untuk menghentikan consumer
    for _ in range(num_consumers):
        q.put(None)

def consumer(q, consumer_id):
    while True:
        item = q.get()
        if item is None:
            print(f"[Consumer-{consumer_id}] Menerima sinyal berhenti. Keluar...")
            break
            
        print(f"[Consumer-{consumer_id}] Mulai memproses tugas {item}...")
        # Menjalankan fungsi CPU-bound
        hasil = proses_berat(item)
        print(f"[Consumer-{consumer_id}] Selesai memproses tugas {item}")

if __name__ == '__main__':
    # Mengganti queue.Queue menjadi multiprocessing.Queue
    task_queue = mp.Queue()
    jumlah_consumer = 3
    jumlah_item = 10
    
    mulai = time.time()

    # Membuat proses Producer
    p_producer = mp.Process(target=producer, args=(task_queue, jumlah_item, jumlah_consumer))
    
    # Membuat proses Consumer
    proses_consumers = []
    for i in range(1, jumlah_consumer + 1):
        p = mp.Process(target=consumer, args=(task_queue, i))
        proses_consumers.append(p)
        
    # Menjalankan semua proses
    p_producer.start()
    for p in proses_consumers:
        p.start()
        
    # Menunggu semua selesai
    p_producer.join()
    for p in proses_consumers:
        p.join()
        
    selesai = time.time()
    print(f"Semua proses selesai. Total Waktu Eksekusi: {selesai - mulai:.4f} detik")

3. Analisis Struktur Kode dan Performa:Apakah pola kode secara struktur banyak berubah?
Secara struktural, kodenya tidak banyak berubah. Logika dasar producer yang mengisi queue, metode consumer mengambil item menggunakan get(), serta teknik penghentian menggunakan sinyal None (Poison Pill) tetap sama persis seperti versi threading. Perubahan utamanya hanya pada pergantian instansiasi queue.Queue menjadi multiprocessing.Queue, penggunaan multiprocessing.Process alih-alih threading.Thread, serta keharusan menggunakan blok perlindungan if __name__ == '__main__': agar sistem operasi bisa dengan aman membuat proses baru.   Perbedaan performa untuk tugas CPU-bound:
Untuk beban kerja komputasi berat (CPU-bound), versi multiprocessing memberikan performa yang jauh lebih cepat dan lebih baik dibandingkan versi threading. Pada versi threading Pertemuan 2, Global Interpreter Lock (GIL) di Python akan menahan eksekusi sehingga perhitungan komputasi sebenarnya tetap berjalan bergantian secara serial pada satu inti (core) CPU, yang justru membuatnya lebih lambat karena overhead perpindahan tugas. Dengan multiprocessing, sistem operasi benar-benar membuat ruang memori dan GIL yang terpisah untuk setiap consumer, sehingga tugas komputasi bisa dieksekusi secara murni paralel pada beberapa core prosesor secara bersamaan.   

4. Refleksi Singkat: Skenario Pemilihan Manager vs Value/ArraySaya akan memilih Manager() ketika membangun aplikasi web scraper paralel yang perlu mengumpulkan dan menyimpan hasil ekstraksi data (misalnya objek JSON atau string) ke dalam sebuah dictionary terpusat. Pada kasus ini, kemudahan penggunaan Manager() untuk menangani struktur data kompleks khas Python jauh lebih menguntungkan, dan kita bisa mentoleransi sedikit penurunan performa akibat overhead komunikasi proksi (IPC).   Sebaliknya, saya akan memilih Value atau Array saat mengembangkan sistem simulasi fisika atau pemrosesan piksel gambar real-time yang melibatkan pembaruan jutaan angka floating-point setiap detiknya. Skenario komputasi berat ini menuntut performa maksimal. Karena data yang diolah hanya berupa angka primitif, menggunakan shared memory langsung dari Value/Array akan jauh lebih cepat karena menghindari bottleneck performa yang ada pada 